<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/MF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import random

In [2]:


content_path = "/content/sample_data/cora.content"

"""
cora.content format:

column 0 → paper ID
column 1–1433 → binary word features
last column → class label
"""

content_path = "/content/sample_data/cora.content"

cora_content = pd.read_csv(content_path, sep="\t", header=None)

paper_ids = cora_content.iloc[:, 0]
features = cora_content.iloc[:, 1:-1]
labels = cora_content.iloc[:, -1]

print("Number of papers:", len(paper_ids))
print("Feature shape:", features.shape)


Number of papers: 2708
Feature shape: (2708, 1433)


In [3]:


"""
cora.cites format:

column 0 → cited paper
column 1 → citing paper
"""

cites_path = "/content/sample_data/cora.cites"

cora_cites = pd.read_csv(cites_path, sep="\t", header=None)
cora_cites.columns = ["cited", "citing"]

print("Number of citation edges:", len(cora_cites))


Number of citation edges: 5429


In [4]:
"""
Map original paper IDs → integer index

Example:
31336 → 0
1061127 → 1
"""

id_map = {id_: i for i, id_ in enumerate(paper_ids)}

cora_cites["cited"] = cora_cites["cited"].map(id_map)
cora_cites["citing"] = cora_cites["citing"].map(id_map)

# Remove any rows with missing IDs
cora_cites = cora_cites.dropna()
cora_cites = cora_cites.astype(int)

In [5]:
N = len(paper_ids)

rows = cora_cites["citing"].values
cols = cora_cites["cited"].values
data = np.ones(len(rows))

# Create sparse adjacency matrix
A = sp.coo_matrix((data, (rows, cols)), shape=(N, N))

print("Adjacency shape:", A.shape)
print("Number of edges:", A.nnz) # number of non Zero elements

Adjacency shape: (2708, 2708)
Number of edges: 5429


In [6]:
A = A + A.T # Make undirected a
A[A > 1] = 1 # Convert 2 to 1

In [7]:
# Convert to edge list
edges = np.array(list(zip(A.nonzero()[0], A.nonzero()[1])))

# Remove duplicate symmetric edges
edges = edges[edges[:,0] < edges[:,1]]

# Train-test split
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

In [8]:
# Create empty adjacency
A_train = sp.lil_matrix((N, N))

for i, j in train_edges:
    A_train[i, j] = 1
    A_train[j, i] = 1

A_train = A_train.tocsr()  # converting to CSR  # Memory Efficient and Faster Multiplication

In [13]:
k = 128  # embedding dimension
         # each node will get a vector of 128 dimensions
"""
TruncatedSVD computes top-k singular values:

A ≈ U_k Σ_k V_k^T
"""

svd = TruncatedSVD(n_components=k, random_state=42)   #approximate reconstruction
Z = svd.fit_transform(A_train)

print("Embedding shape:", Z.shape)

Embedding shape: (2708, 128)


In [9]:
def get_scores(edges, Z):
    scores = []
    for i, j in edges:
        score = np.dot(Z[i], Z[j])
        scores.append(score)
    return np.array(scores)

In [10]:
def generate_negative_edges(num_samples):
    neg_edges = set()

    while len(neg_edges) < num_samples:
        i = random.randint(0, N-1)
        j = random.randint(0, N-1)

        if i != j and A[i, j] == 0:
            neg_edges.add((min(i,j), max(i,j)))

    return np.array(list(neg_edges))

In [27]:
num_runs = 10

auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive test edges
    pos_scores = get_scores(test_edges, Z)

    # Negative test edges (new random ones each run)
    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z)

    # Combine
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_scores = np.concatenate([pos_scores, neg_scores])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Average results
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("Average Matrix Factorization AUC (10 runs):", avg_auc)
print("Average Matrix Factorization AP  (10 runs):", avg_ap)

Average Matrix Factorization AUC (10 runs): 0.7676725529442148
Average Matrix Factorization AP  (10 runs): 0.8237578920650404


In [30]:
"""
==============================
Spectral / Normalized MF
==============================

We factorize:

A_norm = D^{-1/2} A D^{-1/2}

This removes degree bias.
"""

# Compute degrees
degrees = np.array(A_train.sum(axis=1)).flatten()
degrees[degrees == 0] = 1e-10
# D^{-1/2}
deg_inv_sqrt = np.power(degrees, -0.5)
deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0

D_inv_sqrt = sp.diags(deg_inv_sqrt) # diagonal matrix of  degree

# Normalized adjacency
A_norm = D_inv_sqrt @ A_train @ D_inv_sqrt   #Normalization helps remove the effect of high-degree nodes.

# SVD
k = 128
svd = TruncatedSVD(n_components=k, random_state=42)
Z_spec = svd.fit_transform(A_norm)

# Evaluate
# ==============================
# Evaluate Spectral MF (10 runs)
# ==============================

num_runs = 10
auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive edges
    pos_scores = get_scores(test_edges, Z_spec)

    # Random negative edges
    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z_spec)

    # Combine labels
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_scores = np.concatenate([pos_scores, neg_scores])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Average results
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("Spectral MF Average AUC (10 runs):", avg_auc)
print("Spectral MF Average AP  (10 runs):", avg_ap)


Spectral MF Average AUC (10 runs): 0.8014840342343893
Spectral MF Average AP  (10 runs): 0.821799211750881


In [31]:
"""
=====================================
Higher-Order Proximity MF
=====================================

We factorize:

S = A + A^2 + A^3

Captures multi-hop structure.
"""

A_csr = A_train.tocsr()

# Compute powers
A2 = A_csr @ A_csr
A3 = A2 @ A_csr

# Combine
S = A_csr + A2 + A3

# Normalize to avoid large values
S = S / S.max()

# SVD
k = 128
svd = TruncatedSVD(n_components=k, random_state=42)
Z_high = svd.fit_transform(S)

# Evaluate
# ==============================
# Evaluate Higher-Order MF (10 runs)
# ==============================

num_runs = 10
auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive edges
    pos_scores = get_scores(test_edges, Z_high)

    # Random negative edges
    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z_high)

    # Combine
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_scores = np.concatenate([pos_scores, neg_scores])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Compute averages
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("Higher-Order MF Average AUC (10 runs):", avg_auc)
print("Higher-Order MF Average AP  (10 runs):", avg_ap)


Higher-Order MF Average AUC (10 runs): 0.8091675813533058
Higher-Order MF Average AP  (10 runs): 0.839410077601473


In [32]:
"""
=====================================
PMI-based MF (DeepWalk Approximation)
=====================================

DeepWalk ≈ factorizing PMI matrix of random walks.

Steps:
1) T = D^{-1} A
2) M = T + T^2 + ... + T^k
3) Compute PMI
4) SVD
"""

# Transition matrix T = D^{-1} A
degrees = np.array(A_train.sum(axis=1)).flatten()


# Add epsilon to avoid zero division
degrees[degrees == 0] = 1e-10

deg_inv = 1.0 / degrees
D_inv = sp.diags(deg_inv)
T = D_inv @ A_train

# k-step random walk
window_size = 7

M = T.copy()
T_power = T.copy()

for _ in range(window_size - 1):
    T_power = T_power @ T
    M += T_power

M = M.toarray()

# Compute PMI
row_sums = M.sum(axis=1, keepdims=True)
col_sums = M.sum(axis=0, keepdims=True)
total_sum = M.sum()

P_ij = M / total_sum
P_i = row_sums / total_sum
P_j = col_sums / total_sum

# Compute PMI
PMI = np.log((P_ij + 1e-10) / (P_i @ P_j + 1e-10))

# Shifted Positive PMI (like DeepWalk / word2vec)
PMI = PMI - np.log(5)

# Keep only positive values
PMI[PMI < 0] = 0
# SVD
k = 256
svd = TruncatedSVD(n_components=k, random_state=42)
Z_pmi = svd.fit_transform(PMI)

# Evaluate
# ==============================
# Evaluate PMI-based MF (10 runs)
# ==============================

num_runs = 10
auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive edges
    pos_scores = get_scores(test_edges, Z_pmi)

    # Random negative edges
    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z_pmi)

    # Combine labels
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_scores = np.concatenate([pos_scores, neg_scores])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Compute averages
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("PMI-based MF Average AUC (10 runs):", avg_auc)
print("PMI-based MF Average AP  (10 runs):", avg_ap)


PMI-based MF Average AUC (10 runs): 0.8426866319444445
PMI-based MF Average AP  (10 runs): 0.8832285111409013


In [33]:
"""
=========================================
PMI-based Matrix Factorization (Stable)
DeepWalk Approximation
=========================================

DeepWalk ≈ factorizing shifted PMI matrix.

Steps:
1) Compute transition matrix T = D^{-1}A
2) Compute M = T + T^2 + ... + T^window
3) Convert to probability matrix
4) Compute PMI
5) SVD
6) Evaluate
"""

# ---------------------------------------
#  Compute stable transition matrix
# ---------------------------------------

A_csr = A_train.tocsr()

row_sums = np.array(A_csr.sum(axis=1)).flatten()

# Create inverse degree safely
deg_inv = np.zeros_like(row_sums)
nonzero = row_sums > 0
deg_inv[nonzero] = 1.0 / row_sums[nonzero]

D_inv = sp.diags(deg_inv)

# Random walk transition matrix
T = D_inv @ A_csr


# ---------------------------------------
#  Compute multi-step walk matrix
# ---------------------------------------

window_size = 5

M = T.copy()
T_power = T.copy()

for _ in range(window_size - 1):
    T_power = T_power @ T
    M += T_power


# ---------------------------------------
# Convert to probability matrix
# ---------------------------------------

M = M.toarray()

total_sum = M.sum()

# Avoid division by zero
if total_sum == 0:
    total_sum = 1e-9

P_ij = M / total_sum

row_sums = P_ij.sum(axis=1, keepdims=True)
col_sums = P_ij.sum(axis=0, keepdims=True)

# Avoid zero probabilities
row_sums[row_sums == 0] = 1e-9
col_sums[col_sums == 0] = 1e-9


# ---------------------------------------
#  Compute PMI
# ---------------------------------------

PMI = np.log((P_ij + 1e-9) / (row_sums @ col_sums + 1e-9))

# DeepWalk uses Positive PMI
PMI[PMI < 0] = 0


# ---------------------------------------
# SVD Factorization
# ---------------------------------------

k = 128  # try 256 later for improvement

svd = TruncatedSVD(n_components=k, random_state=42)
Z_pmi = svd.fit_transform(PMI)


# ---------------------------------------
# Evaluate
# ---------------------------------------
# ---------------------------------------
# Evaluate (10 runs average)
# ---------------------------------------

num_runs = 10
auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive edges
    pos_scores = get_scores(test_edges, Z_pmi)

    # Random negative edges
    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z_pmi)

    y_true = np.concatenate([
        np.ones(len(pos_scores)),
        np.zeros(len(neg_scores))
    ])

    y_scores = np.concatenate([
        pos_scores,
        neg_scores
    ])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Average results
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("PMI-based MF Average AUC (10 runs):", avg_auc)
print("PMI-based MF Average AP  (10 runs):", avg_ap)


PMI-based MF Average AUC (10 runs): 0.838024779040404
PMI-based MF Average AP  (10 runs): 0.8741801285326785


In [34]:
from sklearn.decomposition import NMF

# -----------------------------
# Non-Negative Matrix Factorization
# -----------------------------

k = 128

# NMF requires a dense non-negative matrix
A_dense = A_train.toarray()

nmf = NMF(
    n_components=k,
    init="random",
    random_state=42,
    max_iter=300
)

W = nmf.fit_transform(A_dense)   # Node embeddings
H = nmf.components_

print("NMF Embedding shape:", W.shape)

# -----------------------------
# Evaluate NMF for link prediction
# -----------------------------
# ==============================
# Evaluate NMF (10 runs)
# ==============================

num_runs = 10
auc_scores = []
ap_scores = []

for _ in range(num_runs):

    # Positive test edges
    pos_scores_nmf = get_scores(test_edges, W)

    # Negative test edges
    neg_edges_nmf = generate_negative_edges(len(test_edges))
    neg_scores_nmf = get_scores(neg_edges_nmf, W)

    # Combine labels
    y_true_nmf = np.concatenate([
        np.ones(len(pos_scores_nmf)),
        np.zeros(len(neg_scores_nmf))
    ])

    y_scores_nmf = np.concatenate([
        pos_scores_nmf,
        neg_scores_nmf
    ])

    auc = roc_auc_score(y_true_nmf, y_scores_nmf)
    ap = average_precision_score(y_true_nmf, y_scores_nmf)

    auc_scores.append(auc)
    ap_scores.append(ap)

# Average results
avg_auc = np.mean(auc_scores)
avg_ap = np.mean(ap_scores)

print("NMF Average AUC (10 runs):", avg_auc)
print("NMF Average AP  (10 runs):", avg_ap)



NMF Embedding shape: (2708, 128)
NMF Average AUC (10 runs): 0.8046149079574152
NMF Average AP  (10 runs): 0.8160370858754554


**Implementation of the paper MF model (Menon & Elkan style)**

In [39]:
num_runs = 10

auc_scores = []
ap_scores = []

for run in range(num_runs):

    print(f"\nRun {run+1}/{num_runs}")

    # ---------------------------
    # Model initialization
    # ---------------------------

    k = 128

    U = nn.Embedding(N, k)
    bias = nn.Embedding(N, 1)

    nn.init.normal_(U.weight, std=0.1)
    nn.init.zeros_(bias.weight)

    optimizer = optim.Adam(list(U.parameters()) + list(bias.parameters()), lr=0.01)
    sigmoid = nn.Sigmoid()

    epochs = 50
    batch_size = 1024

    train_edges_list = train_edges.tolist()

    # ---------------------------
    # Training
    # ---------------------------

    for epoch in range(epochs):

        neg_edges = generate_negative_edges(len(train_edges))

        edges = train_edges_list + neg_edges.tolist()
        labels = np.concatenate([
            np.ones(len(train_edges)),
            np.zeros(len(neg_edges))
        ])

        edges = np.array(edges)

        idx = np.random.permutation(len(edges))
        edges = edges[idx]
        labels = labels[idx]

        for start in range(0, len(edges), batch_size):

            batch_edges = edges[start:start+batch_size]
            batch_labels = torch.tensor(labels[start:start+batch_size]).float()

            i = torch.tensor(batch_edges[:,0])
            j = torch.tensor(batch_edges[:,1])

            ui = U(i)
            uj = U(j)

            bi = bias(i).squeeze()
            bj = bias(j).squeeze()

            score = (ui * uj).sum(dim=1) + bi + bj

            pred = sigmoid(score)

            loss = nn.functional.binary_cross_entropy(pred, batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ---------------------------
    # Evaluation
    # ---------------------------

    Z_final = U.weight.detach().numpy()

    pos_scores = get_scores(test_edges, Z_final)

    neg_edges = generate_negative_edges(len(test_edges))
    neg_scores = get_scores(neg_edges, Z_final)

    y_true = np.concatenate([
        np.ones(len(pos_scores)),
        np.zeros(len(neg_scores))
    ])

    y_scores = np.concatenate([
        pos_scores,
        neg_scores
    ])

    auc = roc_auc_score(y_true, y_scores)
    ap = average_precision_score(y_true, y_scores)

    auc_scores.append(auc)
    ap_scores.append(ap)

    print("Run AUC:", auc)
    print("Run AP :", ap)


# ---------------------------
# Final Average Results
# ---------------------------

print("\n============================")
print("Final MF Average Results")
print("============================")

print("Average AUC:", np.mean(auc_scores))
print("Average AP :", np.mean(ap_scores))




Run 1/10
Run AUC: 0.7631463785583105
Run AP : 0.8201067194576029

Run 2/10
Run AUC: 0.7575138817148761
Run AP : 0.8144000258197499

Run 3/10
Run AUC: 0.764865451388889
Run AP : 0.8154498472344769

Run 4/10
Run AUC: 0.756668244949495
Run AP : 0.815097498201814

Run 5/10
Run AUC: 0.765970249368687
Run AP : 0.81324193439171

Run 6/10
Run AUC: 0.7468532986111112
Run AP : 0.8055501821684946

Run 7/10
Run AUC: 0.7803227588383839
Run AP : 0.8292016218251301

Run 8/10
Run AUC: 0.7555042613636364
Run AP : 0.8090127647102077

Run 9/10
Run AUC: 0.7520463871671257
Run AP : 0.808991184102309

Run 10/10
Run AUC: 0.7600965263429751
Run AP : 0.814853123852928

Final MF Average Results
Average AUC: 0.760298743830349
Average AP : 0.8145904901764423


In [36]:
Z_final = U.weight.detach().numpy()